# فاین‌تیون فارسی Chatterbox — Colab

این نوت‌بوک همان pipeline مخزن را روی Colab یا هر سرور GPU اجاره‌ای اجرا می‌کند.

**نکته‌ی کلیدی درباره‌ی داده:** خروجی preprocessing فقط حدود **۵.۵ کیلوبایت به‌ازای هر کلیپ** است
(≈ ۴۰۰ مگابایت برای ۷۴ هزار کلیپ)، در حالی که فایل‌های صوتی خام ده‌ها گیگابایت‌اند.
پس بهترین کار این است که **preprocessing را محلی انجام دهید** و فقط پوشه‌ی کش را بالا بیاورید.
این هم آپلود را از ده‌ها گیگ به ۴۰۰ مگ می‌رساند و هم ساعت‌های GPU اجاره‌ای را هدر نمی‌دهد.

| مسیر | چه‌کار می‌کند | چه‌وقت |
|---|---|---|
| **الف** | آپلود کش آماده‌ی preprocess | توصیه‌شده — سریع‌ترین |
| **ب** | دانلود دیتاست و preprocess روی Colab | وقتی سیستم محلی GPU ندارد |


## ۱. بررسی GPU

دقت عددی خودکار انتخاب می‌شود: `bf16` روی Ampere به بعد (A100 / L4 / 4090)
و `fp16` روی Turing. T4 معماری Turing است و bf16 سخت‌افزاری ندارد.


In [ ]:
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    total = torch.cuda.get_device_properties(0).total_memory / 2**30
    print(f'{name}  capability {major}.{minor}  {total:.1f} GB')
    print('native bf16:', major >= 8)
else:
    print('No GPU. Runtime > Change runtime type > GPU')


## ۲. آوردن پروژه


In [ ]:
REPO = 'https://github.com/gokhaneraslan/chatterbox-finetuning.git'  # or your fork

import pathlib
if not pathlib.Path('chatterbox-finetuning').exists():
    !git clone -q $REPO
%cd chatterbox-finetuning
!git log --oneline -1


## ۳. نصب وابستگی‌ها

پکیج `chatterbox-tts` عمداً نصب نمی‌شود: سورس آن در `src/chatterbox_` هست و
نصب همزمان دو نسخه را روی مسیر می‌گذارد و معلوم نمی‌شود کدام اجرا می‌شود.


In [ ]:
!pip install -q -r requirements.txt

import importlib
for module in ['torch', 'transformers', 'peft', 'soundfile', 'num2words', 'pyarrow']:
    print(f'{module:14s}', importlib.import_module(module).__version__)


## ۴. توکن HuggingFace

برای دانلود وزن‌ها و دیتاست‌ها. در Colab از پنل 🔑 Secrets استفاده کنید
تا توکن داخل فایل نوت‌بوک ذخیره و به‌اشتراک گذاشته نشود.


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    # Outside Colab, or no secret set: export it before starting instead.
    os.environ.setdefault('HF_TOKEN', '')
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))


## ۵. دریافت وزن‌های مدل (~۳.۲ گیگ)

نسخه‌ها از `versions.lock.json` خوانده می‌شوند، پس هر اجرا دقیقاً همان artifactها را می‌گیرد.
این مرحله توکنایزر `[fa]` را هم می‌سازد.


In [ ]:
!python tools/fetch_models.py
!ls -la pretrained_models/


## ۶الف. مسیر توصیه‌شده — آپلود کش preprocess

روی سیستم محلی:

```bash
python tools/build_dataset.py --sources mana narration yoda --dedupe
python -m src.preprocess_ljspeech
tar -czf preprocess_fa.tar.gz -C MyTTSDataset preprocess
```

فایل حاصل را در Google Drive بگذارید، بعد این دو سلول را اجرا کنید.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
CACHE_ARCHIVE = '/content/drive/MyDrive/preprocess_fa.tar.gz'

import pathlib
if pathlib.Path(CACHE_ARCHIVE).exists():
    !mkdir -p MyTTSDataset
    !tar -xzf $CACHE_ARCHIVE -C MyTTSDataset
    n = len(list(pathlib.Path('MyTTSDataset/preprocess').glob('*.pt')))
    print(f'{n:,} preprocessed clips ready')
else:
    print(CACHE_ARCHIVE, 'not found - mount Drive, or use path B below.')


## ۶ب. مسیر جایگزین — ساخت دیتاست روی Colab

فقط وقتی سیستم محلی GPU ندارد. YodaLingua حدود ۱ گیگ دانلود و
۱۲ گیگ فضای دیسک برای wav می‌خواهد، و preprocessing هم زمان‌بر است.


In [ ]:
# Skip entirely if you used path A.
!python tools/fetch_datasets.py yoda
!python tools/build_dataset.py --sources yoda --dedupe
!python -m src.preprocess_ljspeech


## ۷. آموزش

اول یک smoke-test کوتاه، تا پیش از شروع اجرای چندساعته مطمئن شویم مسیر سالم است
و ببینیم چقدر VRAM مصرف می‌شود.


In [ ]:
!python train.py --max-steps 5 --batch-size 4 --grad-accum 1 --workers 2 --no-preprocess


اگر پیک VRAM خیلی کمتر از ظرفیت کارت بود، `--batch-size` را بالا ببرید:
روی T4 (۱۶ گیگ) معمولاً ۱۶ و روی A100 یا 4090 حدود ۳۲ جا می‌شود.

برای اجرای کامل `--max-steps` را بردارید و `--grad-accum` را طوری بگذارید که
batch مؤثر (`batch_size × grad_accum`) حدود ۳۲ بماند.


In [ ]:
!python train.py --batch-size 16 --grad-accum 2 --workers 2 --no-preprocess


### پایش با TensorBoard


In [ ]:
%load_ext tensorboard
%tensorboard --logdir chatterbox_output/runs


## ۸. تولید صدا

`--long` متن را روی مرزهای جمله‌ی فارسی می‌شکند و تکه‌ها را به هم می‌چسباند.
بدون آن هر فراخوانی حداکثر حدود ۴۰ ثانیه صدا می‌دهد (سقف ۱۰۰۰ توکن گفتاری در ۲۵ هرتز).


In [ ]:
TEXT = 'سلام، این یک آزمایش برای مدل گفتار فارسی است. امیدوارم صدای طبیعی و روانی داشته باشد.'

!python infer_fa.py --text "$TEXT" --out sample.wav

from IPython.display import Audio, display
display(Audio('sample.wav'))


In [ ]:
long_text = '''هوش مصنوعی در سال‌های اخیر پیشرفت چشمگیری داشته است.
مدل‌های زبانی بزرگ توانسته‌اند در ترجمه، خلاصه‌سازی و تولید متن به نتایج قابل توجهی برسند؛
با این حال چالش‌هایی مانند سوگیری و نیاز به داده‌های باکیفیت همچنان باقی است.'''

open('long.txt', 'w', encoding='utf-8').write(long_text)
!python infer_fa.py --text-file long.txt --long --out long.wav

from IPython.display import Audio, display
display(Audio('long.wav'))


## ۹. ذخیره‌ی نتیجه

آداپتور LoRA چند ده مگابایت است و به همان مدل پایه‌ای که رویش آموزش دیده گره خورده؛
روی مدل پایه‌ی دیگری بارگذاری نمی‌شود.


In [ ]:
!du -sh chatterbox_output/persian_adapter
!cp -r chatterbox_output/persian_adapter /content/drive/MyDrive/
print('saved to Drive')


---

### Colab یا اجاره‌ی GPU؟

| | Colab Pro | Colab Pro+ | RunPod / Vast (4090) |
|---|---|---|---|
| GPU | T4 ۱۶ گیگ، گاهی L4 | گاهی A100 | 4090 ۲۴ گیگ |
| bf16 سخت‌افزاری | ندارد (T4 = Turing) | دارد (A100) | دارد |
| محدودیت زمان | ~۱۲ ساعت، با قطعی | ~۲۴ ساعت | ندارد |
| دیسک ماندگار | ندارد | ندارد | volume دارد |
| هزینه | ~۱۰$ ماهانه | ~۵۰$ ماهانه | ~۰.۴$ ساعتی |

برای یک اجرای کامل، اجاره‌ی 4090 معمولاً هم ارزان‌تر درمی‌آید و هم قطع نمی‌شود.
Colab برای آزمایش سریع و تنظیم hyperparameter عالی است.
